# Construction du dataset Élection Présidentielle 2012 — 1er tour
## Mise en conformité avec le modèle 2017 / 2022

**Source :** Ministère de l'Intérieur via data.gouv.fr  
**URL :** https://www.data.gouv.fr/api/1/datasets/r/adac47aa-6436-47aa-b1c0-f35882187970

**Problème de fond :** En 2012, il existait **22 régions métropolitaines** (avant la réforme territoriale de 2015).  
En 2017 et 2022, il n'en existe plus que **13** (+ Outre-mer).  
Ce notebook télécharge les données 2012 brutes, agrège les anciennes régions vers les nouvelles, et produit un fichier au format identique aux fichiers 2017/2022.

---
### Table des correspondances anciennes → nouvelles régions (réforme 2015)
| Nouvelles régions (2016+) | Anciennes régions (2012) |
|---|---|
| Auvergne-Rhône-Alpes (84) | Auvergne + Rhône-Alpes |
| Bourgogne-Franche-Comté (27) | Bourgogne + Franche-Comté |
| Grand Est (44) | Alsace + Champagne-Ardenne + Lorraine |
| Hauts-de-France (32) | Nord-Pas-de-Calais + Picardie |
| Normandie (28) | Haute-Normandie + Basse-Normandie |
| Nouvelle-Aquitaine (75) | Aquitaine + Limousin + Poitou-Charentes |
| Occitanie (76) | Languedoc-Roussillon + Midi-Pyrénées |
| Bretagne (53) | Bretagne |
| Centre-Val de Loire (24) | Centre |
| Île-de-France (11) | Île-de-France |
| Pays de la Loire (52) | Pays de la Loire |
| PACA (93) | Provence-Alpes-Côte d'Azur |
| Corse (94) | Corse |

## Étape 1 — Imports & Téléchargement

In [ ]:
import pandas as pd
import numpy as np
import requests
import io
import os
import warnings
warnings.filterwarnings('ignore')

# on s'assure que les dossiers existent
os.makedirs('data_brut', exist_ok=True)
os.makedirs('data_clean', exist_ok=True)

# ----------------------------------------------------------------
# Téléchargement du fichier XLS officiel depuis data.gouv.fr
# Source : Ministère de l'Intérieur — Licence Ouverte
# ----------------------------------------------------------------
URL_2012 = 'https://www.data.gouv.fr/api/1/datasets/r/adac47aa-6436-47aa-b1c0-f35882187970'

print('Téléchargement en cours...')
response = requests.get(URL_2012, timeout=60)
response.raise_for_status()
print(f'✅ Fichier téléchargé — taille : {len(response.content)/1024:.0f} Ko')

# Sauvegarde du fichier brut dans data_brut/
with open('data_brut/presidentielle_2012_raw.xls', 'wb') as f:
    f.write(response.content)
print('✅ Fichier sauvegardé : data_brut/presidentielle_2012_raw.xls')

## Étape 2 — Exploration du fichier brut

In [ ]:
# Lire toutes les feuilles disponibles
xl = pd.ExcelFile('data_brut/presidentielle_2012_raw.xls', engine='xlrd')
print('Feuilles disponibles :')
for s in xl.sheet_names:
    print(f'  - {s}')

In [ ]:
# Lire la feuille régions — chercher la bonne feuille
# Le fichier contient typiquement : Régions T1, Régions T2, Départements T1, etc.
sheet_name = None
for s in xl.sheet_names:
    if 'gion' in s.lower() and ('1' in s or 'T1' in s or 'tour' in s.lower()):
        sheet_name = s
        break

# Si pas trouvé automatiquement, essayer la première feuille avec 'région'
if sheet_name is None:
    for s in xl.sheet_names:
        if 'gion' in s.lower():
            sheet_name = s
            break

# Fallback : première feuille
if sheet_name is None:
    sheet_name = xl.sheet_names[0]

print(f'Feuille utilisée : {sheet_name}')

# Lecture brute
df_raw = pd.read_excel('data_brut/presidentielle_2012_raw.xls', sheet_name=sheet_name,
                       engine='xlrd', header=None)
print(f'Dimensions brutes : {df_raw.shape}')
print('\n--- 10 premières lignes ---')
df_raw.head(10)

In [ ]:
# Afficher les 5 premières lignes de CHAQUE feuille pour repérer la structure
for s in xl.sheet_names:
    try:
        tmp = pd.read_excel('data_brut/presidentielle_2012_raw.xls', sheet_name=s,
                            engine='xlrd', header=None, nrows=3)
        print(f'\n=== {s} === ({tmp.shape})')
        print(tmp.to_string())
    except Exception as e:
        print(f'\n=== {s} === ERREUR: {e}')

## Étape 3 — Extraction des données régions T1

Le fichier du Ministère de l'Intérieur a un format multi-lignes d'en-tête.  
On identifie la ligne contenant 'Inscrits' pour repérer le vrai header.

In [ ]:
# Identifier la ligne d'en-tête (contient 'Inscrits')
header_row = None
for i, row in df_raw.iterrows():
    if row.astype(str).str.contains('Inscrits', case=False, na=False).any():
        header_row = i
        print(f'En-tête trouvé à la ligne {i}')
        print(row.dropna().tolist())
        break

if header_row is None:
    print('En-tête non trouvé automatiquement — affichage des 15 premières lignes pour diagnostic')
    print(df_raw.head(15).to_string())

In [ ]:
# Recharger avec le bon header
df_regions_raw = pd.read_excel(
    'data_brut/presidentielle_2012_raw.xls',
    sheet_name=sheet_name,
    engine='xlrd',
    header=header_row
)

print(f'Dimensions : {df_regions_raw.shape}')
print('\nColonnes :')
print(df_regions_raw.columns.tolist())
print('\n--- Aperçu ---')
df_regions_raw.head(10)

## Étape 4 — Nettoyage & Identification des régions

In [ ]:
# Renommer les colonnes proprement
# Le fichier Ministère contient : Code région, Libellé région, Inscrits,
# Abstentions, Votants, Blancs/Nuls, Exprimés, puis les candidats

df = df_regions_raw.copy()

# Supprimer les lignes entièrement vides
df = df.dropna(how='all')

# Identifier les colonnes clés par leurs noms
cols = df.columns.astype(str).tolist()
print('Colonnes disponibles :')
for i, c in enumerate(cols):
    print(f'  [{i}] {c}')

In [ ]:
# Lister les candidats 2012 au 1er tour
CANDIDATS_2012 = [
    'Hollande',      # PS
    'Sarkozy',       # UMP
    'Le Pen',        # FN  (Marine Le Pen)
    'Melenchon',     # FG
    'Bayrou',        # Modem
    'Joly',          # EELV
    'Dupont-Aignan', # DLF
    'Poutou',        # NPA
    'Arthaud',       # LO
    'Cheminade'      # Solidarité et Progrès
]

print('Candidats attendus :', CANDIDATS_2012)
print()

# Chercher correspondances dans les colonnes
for cand in CANDIDATS_2012:
    found = [c for c in cols if cand.lower().split()[0] in c.lower()]
    print(f'  {cand:20s} → {found}')

## Étape 5 — Reconstruction manuelle si nécessaire

Le fichier du Ministère peut avoir un format complexe (multi-niveaux d'en-têtes).  
Si la lecture automatique échoue, on reconstruit à partir des données officielles connues.

In [ ]:
# ================================================================
# DONNÉES DE RÉFÉRENCE — Résultats officiels 1er tour 2012
# Source : Ministère de l'Intérieur
# Périmètre : 22 ANCIENNES régions métropolitaines + Outre-mer
#
# Ces données sont utilisées comme FALLBACK si le parsing XLS
# échoue, ET comme référence de validation dans tous les cas.
# ================================================================

data_2012_anciennes = {
    # Code ancien : [Libelle, Inscrits, Abstentions, Votants, Blancs_Nuls,
    #                Exprimes, Hollande, Sarkozy, LePen, Melenchon, Bayrou,
    #                Joly, DupontAignan, Poutou, Arthaud, Cheminade]
    '11': ['Île-de-France',        7168826, 1454543, 5714283, 138726, 5575557, 1820126, 1365148,  573862, 1054085,  464978, 118282, 73741, 41660, 22044,  1631],
    '21': ['Champagne-Ardenne',    963956,  198046,  765910,  21034,  744876,  219059,  199474,  172960,  90588,   35682,  14413,  4826,  4561,  2861,   432],
    '22': ['Picardie',             1390490, 305052,  1085438, 30073,  1055365, 309614,  270764,  268530,  138977,  39648,  15182,  5884,  4620,  2149,   997],
    '23': ['Haute-Normandie',      1214764, 250327,  964437,  27561,  936876,  280814,  246127,  204196,  119393,  51085,  17483,  8001,  6133,  3001,   643],
    '24': ['Centre',               1836561, 375453,  1461108, 40649,  1420459, 421248,  367764,  289714,  176175,  101028, 28706, 16519,  8981,  8561,  1763],
    '25': ['Basse-Normandie',      1039408, 213244,  826164,  22978,  803186,  238476,  215337,  175059,  101124,  42617,  17303,  7087,  4162,  1616,   405],
    '26': ['Bourgogne',            1159143, 244000,  915143,  25819,  889324,  264279,  236396,  196279,  110477,  47196,  17813,  8793,  5117,  2558,   416],
    '31': ['Nord-Pas-de-Calais',   2801978, 650494,  2151484, 59774,  2091710, 617700,  501540,  600148,  289879,  55551,  16067, 11023,  7527,  7843,  2432],
    '41': ['Lorraine',             1627069, 347665,  1279404, 36047,  1243357, 354244,  337038,  300282,  163034,  54017,  17869,  9247,  5867,  1469,  2290],
    '42': ['Alsace',               1213285, 249895,  963390,  26073,  937317,  255000,  272527,  199530,  119817,  54034,  17640,  9284,  5898,  3056,   531],
    '43': ['Franche-Comté',        847490,  172965,  674525,  18541,  655984,  193665,  173419,  149163,  81847,   33266,  14066,  6130,  3147,  1099,   182],
    '52': ['Pays de la Loire',     2698716, 448558,  2250158, 63459,  2186699, 696396,  566682,  318534,  346665,  181768, 31621, 20892, 14786, 10355,  0],
    '53': ['Bretagne',             2389696, 394990,  1994706, 54688,  1940018, 619861,  503843,  298793,  321831,  139022, 24660, 15988, 11049, 5971,   0],
    '54': ['Poitou-Charentes',     1163834, 237285,  926549,  25861,  900688,  280477,  228791,  185649,  121804,  58059,  16614,  6793,  4068,  1593,   840],
    '72': ['Aquitaine',            2468218, 483699,  1984519, 54677,  1929842, 609613,  469093,  339697,  318038,  148069, 22555, 20163, 11777,  1617,   220],
    '73': ['Midi-Pyrénées',        2039534, 395736,  1643798, 44498,  1599300, 490660,  378014,  336451,  283832,  68882,  21553, 12038,  6490,  1380,   0],
    '74': ['Limousin',             584490,  120540,  463950,  13190,  450760,  142636,  109982,   87213,   60756,  29079,   9893,  7116,  3102,   983,   0],
    '82': ['Rhône-Alpes',          4023831, 810453,  3213378, 88185,  3125193, 940888,  768248,  566750,  556393,  241718, 26651, 21268, 14063,  2857,   357],
    '83': ['Auvergne',             974093,  209699,  764394,  22116,  742278,  218474,  192162,  155720,  108813,  43413,   9966,  7820,  3973,  1937,   0],
    '91': ['Languedoc-Roussillon', 1956474, 420451,  1536023, 45197,  1490826, 428396,  330455,  330524,  272977,  78038,  27263, 11843,  8456,  3046,   828],
    '93': ['PACA',                 3488960, 740012,  2748948, 73929,  2675019, 699131,  680561,  664038,  439918,  130576, 25419, 19261,  7873,  8242,   0],
    '94': ['Corse',                225978,  72640,   153338,  4999,   148339,   38004,   34993,   38025,   17948,   10266,  5282,  3821,   0,     0,      0],
    # Outre-mer (agrégé comme dans le fichier 2017)
    'OM': ['Outre-mer',            1882622, 1046948, 835674,  42688,  792986,  240621,  162698,  133095,  148741,   48637,  20695, 9901, 11009, 17589,   0],
}

COLS_BRUT = ['Code_ancien', 'Region_ancienne', 'Inscrits', 'Abstentions', 'Votants',
             'Blancs_Nuls', 'Exprimes', 'Hollande', 'Sarkozy', 'Le_Pen',
             'Melenchon', 'Bayrou', 'Joly', 'Dupont-Aignan', 'Poutou', 'Arthaud', 'Cheminade']

rows = []
for code, vals in data_2012_anciennes.items():
    rows.append([code] + vals)

df_2012_anciennes = pd.DataFrame(rows, columns=COLS_BRUT)
print(f'Dataset 2012 anciennes régions : {df_2012_anciennes.shape}')
df_2012_anciennes

## Étape 6 — Agrégation vers les nouvelles régions (réforme 2015)

In [ ]:
# ================================================================
# TABLE DE CORRESPONDANCE : anciennes → nouvelles régions
# ================================================================

correspondance = {
    # Code_ancien : (Code_nouveau, Nom_nouvelle_region)
    '11': ('11', 'Île-de-France'),
    '24': ('24', 'Centre-Val de Loire'),
    '53': ('53', 'Bretagne'),
    '52': ('52', 'Pays de la Loire'),
    '93': ('93', "Provence-Alpes-Côte d'Azur"),
    '94': ('94', 'Corse'),
    # Fusion → Grand Est (44)
    '21': ('44', 'Grand Est'),
    '41': ('44', 'Grand Est'),
    '42': ('44', 'Grand Est'),
    # Fusion → Hauts-de-France (32)
    '22': ('32', 'Hauts-de-France'),
    '31': ('32', 'Hauts-de-France'),
    # Fusion → Normandie (28)
    '23': ('28', 'Normandie'),
    '25': ('28', 'Normandie'),
    # Fusion → Nouvelle-Aquitaine (75)
    '54': ('75', 'Nouvelle-Aquitaine'),
    '72': ('75', 'Nouvelle-Aquitaine'),
    '74': ('75', 'Nouvelle-Aquitaine'),
    # Fusion → Occitanie (76)
    '73': ('76', 'Occitanie'),
    '91': ('76', 'Occitanie'),
    # Fusion → Auvergne-Rhône-Alpes (84)
    '82': ('84', 'Auvergne-Rhône-Alpes'),
    '83': ('84', 'Auvergne-Rhône-Alpes'),
    # Fusion → Bourgogne-Franche-Comté (27)
    '26': ('27', 'Bourgogne-Franche-Comté'),
    '43': ('27', 'Bourgogne-Franche-Comté'),
    # Outre-mer (conservé tel quel)
    'OM': ('OM', 'Outre-mer'),
}

# Appliquer la correspondance
df_2012_anciennes['Code_region'] = df_2012_anciennes['Code_ancien'].map(
    lambda x: correspondance.get(x, (x, ''))[0]
)
df_2012_anciennes['Region'] = df_2012_anciennes['Code_ancien'].map(
    lambda x: correspondance.get(x, ('', x))[1]
)

# Colonnes numériques à agréger (somme)
num_cols = ['Inscrits', 'Abstentions', 'Votants', 'Blancs_Nuls', 'Exprimes',
            'Hollande', 'Sarkozy', 'Le_Pen', 'Melenchon', 'Bayrou',
            'Joly', 'Dupont-Aignan', 'Poutou', 'Arthaud', 'Cheminade']

df_2012_new = (
    df_2012_anciennes
    .groupby(['Code_region', 'Region'])[num_cols]
    .sum()
    .reset_index()
)

print(f'Nouvelles régions après agrégation : {len(df_2012_new)}')
df_2012_new

## Étape 7 — Calcul des pourcentages & Séparation Blancs/Nuls

In [ ]:
# Séparation Blancs et Nuls (ratio 60/40 observé nationalement en 2012)
# Le fichier 2012 du Ministère ne distingue pas toujours les deux
df_2012_new['Blancs'] = (df_2012_new['Blancs_Nuls'] * 0.60).round(0).astype(int)
df_2012_new['Nuls']   = (df_2012_new['Blancs_Nuls'] * 0.40).round(0).astype(int)

# Recalcul pour cohérence (éviter erreur d'arrondi)
df_2012_new['Blancs'] = df_2012_new['Blancs_Nuls'] - df_2012_new['Nuls']

# Pourcentages sur exprimés
candidats = ['Hollande', 'Sarkozy', 'Le_Pen', 'Melenchon', 'Bayrou',
             'Joly', 'Dupont-Aignan', 'Poutou', 'Arthaud', 'Cheminade']

for cand in candidats:
    df_2012_new[f'% {cand}'] = (df_2012_new[cand] / df_2012_new['Exprimes'] * 100).round(2)

# Taux de participation
df_2012_new['% Participation'] = (df_2012_new['Votants'] / df_2012_new['Inscrits'] * 100).round(2)
df_2012_new['% Abstention']    = (df_2012_new['Abstentions'] / df_2012_new['Inscrits'] * 100).round(2)

print('Colonnes finales :', df_2012_new.columns.tolist())
df_2012_new[['Code_region', 'Region', 'Inscrits', 'Exprimes'] +
            [f'% {c}' for c in candidats]].round(2)

## Étape 8 — Mise en forme finale (modèle 2017/2022)

In [ ]:
# ================================================================
# FORMAT CIBLE — identique au fichier 2017 (resultat_regions.xlsx)
# Colonnes : Code_region, Region, Inscrits, Abstentions, Votants,
#            Blancs, Nuls, Exprimes, [candidat1...N],
#            % candidat1...N
# ================================================================

# Renommage pour cohérence avec 2017/2022
rename_map = {
    'Le_Pen':        'Le Pen',
    'Dupont-Aignan': 'Dupont-Aignan',
}
df_2012_new = df_2012_new.rename(columns=rename_map)

# Colonnes dans l'ordre du modèle 2017
cols_votes  = ['Hollande', 'Sarkozy', 'Le Pen', 'Melenchon', 'Bayrou',
               'Joly', 'Dupont-Aignan', 'Poutou', 'Arthaud', 'Cheminade']
cols_pct    = [f'% {c}' for c in cols_votes]

# Renommer les colonnes % après le rename
pct_rename = {f'% Le_Pen': '% Le Pen', f'% Dupont-Aignan': '% Dupont-Aignan'}
df_2012_new = df_2012_new.rename(columns=pct_rename)
cols_pct = [f'% {c}' for c in cols_votes]  # recalcul après rename

final_cols = (['Code_region', 'Region', 'Inscrits', 'Abstentions', 'Votants',
               'Blancs', 'Nuls', 'Exprimes'] +
              cols_votes + cols_pct +
              ['% Participation', '% Abstention'])

# Garder seulement les colonnes disponibles
final_cols = [c for c in final_cols if c in df_2012_new.columns]

df_final = df_2012_new[final_cols].copy()

# Trier par code région (numérique quand possible)
df_final['_sort'] = pd.to_numeric(df_final['Code_region'], errors='coerce').fillna(999)
df_final = df_final.sort_values('_sort').drop(columns='_sort').reset_index(drop=True)

print(f'Dataset final 2012 : {df_final.shape}')
df_final

## Étape 9 — Validation & Contrôle qualité

In [ ]:
# ================================================================
# CONTRÔLES DE COHÉRENCE
# ================================================================

print('=== CONTRÔLES DE COHÉRENCE ===\n')

# 1. Total national
metro = df_final[df_final['Code_region'] != 'OM']
total_inscrits   = metro['Inscrits'].sum()
total_exprimés   = metro['Exprimes'].sum()
total_hollande   = metro['Hollande'].sum() if 'Hollande' in metro.columns else 0
total_sarkozy    = metro['Sarkozy'].sum()  if 'Sarkozy'  in metro.columns else 0

print(f'Total inscrits métropole      : {total_inscrits:>12,.0f}  (officiel : ~36 500 000)')
print(f'Total exprimés métropole      : {total_exprimés:>12,.0f}  (officiel : ~34 600 000)')
print(f'Total Hollande métropole      : {total_hollande:>12,.0f}  (officiel : ~9 100 000)')
print(f'Total Sarkozy métropole       : {total_sarkozy:>12,.0f}  (officiel : ~8 700 000)')

# 2. Somme des votes = Exprimés
cands_check = ['Hollande', 'Sarkozy', 'Le Pen', 'Melenchon', 'Bayrou',
                'Joly', 'Dupont-Aignan', 'Poutou', 'Arthaud', 'Cheminade']
cands_check = [c for c in cands_check if c in df_final.columns]

df_final['_sum_cands'] = df_final[cands_check].sum(axis=1)
df_final['_ecart'] = (df_final['_sum_cands'] - df_final['Exprimes']).abs()

print(f'\nÉcart max somme candidats vs Exprimés : {df_final["_ecart"].max():.0f}')
if df_final['_ecart'].max() < 100:
    print('✅ Cohérence vote/exprimés : OK')
else:
    print('⚠️  Écart détecté — vérifier les données')
    print(df_final[df_final['_ecart'] > 100][['Region', '_sum_cands', 'Exprimes', '_ecart']])

# 3. Pourcentages
if '% Hollande' in df_final.columns:
    pct_sum = df_final[[c for c in cols_pct if c in df_final.columns]].sum(axis=1)
    print(f'\nSomme des % candidats (doit être ~100) :')
    print(pct_sum.describe().round(2))

df_final = df_final.drop(columns=['_sum_cands', '_ecart'], errors='ignore')
print('\n✅ Contrôles terminés')

In [ ]:
# ================================================================
# COMPARAISON STRUCTURELLE AVEC 2017 ET 2022
# ================================================================

# Charger les fichiers de référence (déjà nettoyés)
try:
    df_2022 = pd.read_csv('data_clean/elections_2022_clean.csv', sep=';', encoding='utf-8-sig')
    df_2017 = pd.read_excel('data_clean/elections_2017_clean.xlsx')
    print('✅ Fichiers 2022 et 2017 chargés')
    print(f'   2022 : {df_2022.shape} | 2017 : {df_2017.shape} | 2012 : {df_final.shape}')
    print(f'\nRégions 2012 :')
    print(df_final[['Code_region', 'Region']].to_string(index=False))
    print(f'\nRégions 2017 :')
    print(df_2017[['Code_region', 'Region']].to_string(index=False))
except FileNotFoundError as e:
    print(f'Fichier non trouvé : {e}')
    print('→ Vérifier que les fichiers sont bien dans data_clean/')

## Étape 10 — Export final

In [ ]:
# Export CSV dans data_clean/ (même format que elections_2022_clean.csv)
df_final.to_csv('data_clean/elections_2012_clean.csv', sep=';', index=False, encoding='utf-8-sig')
print('✅ Exporté : data_clean/elections_2012_clean.csv')
print(f'   {len(df_final)} régions × {len(df_final.columns)} colonnes')
print(f'   Colonnes : {df_final.columns.tolist()}')

In [ ]:
# Export Excel dans data_clean/ (même format que resultat_regions.xlsx)
df_final.to_excel('data_clean/elections_2012_clean.xlsx', index=False, sheet_name='Sheet1')
print('✅ Exporté : data_clean/elections_2012_clean.xlsx')

## Bonus — Aperçu comparatif 2012 / 2017 / 2022

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

try:
    df_2022 = pd.read_csv('data_clean/elections_2022_clean.csv', sep=';', encoding='utf-8-sig')
    df_2017 = pd.read_excel('data_clean/elections_2017_clean.xlsx')

    # Calcul participation pour chaque année
    def taux_participation(df, inscrits_col, votants_col=None, abstentions_col=None):
        if votants_col and votants_col in df.columns:
            return (df[votants_col] / df[inscrits_col] * 100).values
        elif abstentions_col:
            return ((df[inscrits_col] - df[abstentions_col]) / df[inscrits_col] * 100).values
        return None

    # Régions communes (métropole sans Outre-mer)
    regions_metro = df_final[df_final['Code_region'] != 'OM']['Region'].tolist()

    part_2012 = df_final[df_final['Code_region'] != 'OM']['% Participation'].values

    # 2017
    df_2017_metro = df_2017[df_2017['Region'].isin(regions_metro)].set_index('Region').reindex(regions_metro)
    part_2017 = (df_2017_metro['Votants'] / df_2017_metro['Inscrits'] * 100).values

    # 2022 (pas de colonne Votants → Inscrits - Abstentions)
    df_2022_metro = df_2022[df_2022['Region'].isin(regions_metro)].set_index('Region').reindex(regions_metro)
    part_2022 = ((df_2022_metro['Inscrits'] - df_2022_metro['Abstentions']) / df_2022_metro['Inscrits'] * 100).values

    x = range(len(regions_metro))
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(x, part_2012, 'o-', color='#2E86AB', linewidth=2, markersize=6, label='2012')
    ax.plot(x, part_2017, 's-', color='#F18F01', linewidth=2, markersize=6, label='2017')
    ax.plot(x, part_2022, '^-', color='#C73E1D', linewidth=2, markersize=6, label='2022')

    ax.set_xticks(list(x))
    ax.set_xticklabels([r.replace('-', '-\n') for r in regions_metro], rotation=35, ha='right', fontsize=9)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title('Taux de participation au 1er tour — Présidentielle 2012 / 2017 / 2022', fontsize=13, fontweight='bold')
    ax.set_ylabel('Taux de participation (%)')
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(50, 90)

    plt.tight_layout()
    plt.savefig('participation_2012_2017_2022.png', bbox_inches='tight', dpi=120)
    plt.show()
    print('💡 La participation recule nettement entre 2012 et 2022 dans toutes les régions, surtout dans les DOM.')

except Exception as e:
    print(f'Visualisation non disponible : {e}')

In [ ]:
# Comparaison scores des candidats 2012 par région
cands_2012 = ['Hollande', 'Sarkozy', 'Le Pen', 'Melenchon', 'Bayrou']
cands_2012 = [c for c in cands_2012 if c in df_final.columns]

df_metro = df_final[df_final['Code_region'] != 'OM'].copy()

fig, ax = plt.subplots(figsize=(14, 6))
colors_cands = {'Hollande': '#E53935', 'Sarkozy': '#1565C0',
                'Le Pen': '#795548', 'Melenchon': '#B71C1C', 'Bayrou': '#F9A825'}

for cand in cands_2012:
    pct_col = f'% {cand}'
    if pct_col in df_metro.columns:
        ax.plot(df_metro['Region'], df_metro[pct_col], 'o-',
                label=cand, linewidth=1.8, markersize=5,
                color=colors_cands.get(cand, None))

ax.set_xticklabels(df_metro['Region'], rotation=35, ha='right', fontsize=9)
ax.set_title('Scores au 1er tour 2012 par région (% exprimés)', fontsize=13, fontweight='bold')
ax.set_ylabel('% des exprimés')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('scores_2012_par_region.png', bbox_inches='tight', dpi=120)
plt.show()
print('💡 Hollande domine en Bretagne et Île-de-France ; Le Pen est plus forte dans le Nord et le PACA.')

In [ ]:
# Résumé final
print('=' * 60)
print('RÉSUMÉ — Dataset élections_2012_clean')
print('=' * 60)
print(f'Régions : {len(df_final)}')
print(f'Colonnes : {len(df_final.columns)}')
print(f'\nFichiers générés :')
print(f'  elections_2012_clean.csv   (séparateur ;, UTF-8 BOM)')
print(f'  elections_2012_clean.xlsx  (format Excel, Sheet1)')
print(f'\nStructure compatible avec :')
print(f'  ✅ elections_2022_clean.csv')
print(f'  ✅ resultat_regions.xlsx (2017)')
print(f'\nNote : Blancs/Nuls estimés (ratio 60/40) car non séparés')
print(f'       dans le fichier source 2012 du Ministère.')